[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-10.ipynb)

# 10주차 실습: 정렬 목적함수와 GRPO 보상 broadcast 관찰

- 목표: SFT·선호·PPO·DPO의 준비된 목적함수 예제를 비교한 뒤 outcome GRPO의 숫자 관찰
- 환경: Python 3, CPU. 산술 관찰은 표준 라이브러리, 실제 후보 수집은 PyTorch·Transformers 사용
- 코드 변경 과제 없음. 모든 셀을 순서대로 실행
- 실제 모델 학습·rollout·verifier 실행 없음. 후보 응답 수집에서는 tokenizer와 생성 실행
- reward와 길이는 교수 제공 산술 예시. 출력은 학습 성능 측정값이 아님
- 수업계획서와 이슈 #4에 따른 관찰 전용 실습

## 출처와 공개 정보 맥락

- GRPO: [DeepSeekMath 4.1.2](https://arxiv.org/html/2402.03300v3#S4.SS1.SSS2)
- 집계: [DAPO 3.3](https://arxiv.org/html/2503.14476v1#S3.SS3)
- 제주 공개 정보: [위키백과 「제주특별자치도」 고정판 42464087](https://ko.wikipedia.org/w/index.php?title=제주특별자치도&oldid=42464087), 7주차와 동일 출처
- 질문 "제주특별자치도의 두 행정시는?": 제주시·서귀포시의 정적 정보와 최신 운영정보의 차이 검토
- 원문은 실행 중 다운로드. 앞의 관광 응대 후보는 교수 작성 예시, §4-1의 생성 결과는 별도 실제 수집 기록


## 학습 기록 조건

- 학번: 202221015. GitHub 과제 제출용 관찰 기록.
- 관찰형 주차로 제공된 계산과 실험 조건 유지, 학생 알고리즘이나 verifier 추가 없음.
- 교수 작성 예시와 실제 공개 원문 확인, 실제 모델 생성 여부를 구분하여 기록.
- 한국어 위키백과 기여자들의 [제주특별자치도 고정판 42464087](https://ko.wikipedia.org/w/index.php?oldid=42464087), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). 원문 전체 파일 저장 없음.
- 생성형 AI의 코드 실행 및 결과 정리 보조 사용. 학습자 개인의 이해도를 평가한 기록으로 주장하지 않음.
- 원본 코드의 연락처가 포함된 User-Agent만 공개 강의 저장소 URL로 변경.
- 재현 보완: 수집 API의 고정판 번호와 제목 일치 검사 추가. raw prompt, CPU 2 threads, float32, eager attention, seed 2026, 24토큰 상한과 공유 EOS 조건은 원본 그대로 유지.
- 런타임 호환 보완: Transformers 4.57.6이 GenerationConfig의 전역 기본값과 같은 값을 모델 기본값으로 덮어쓰는 동작을 확인. generate()에 use_model_defaults=False와 do_sample=False를 명시하여 원본이 의도한 동일한 탐욕 생성 조건을 강제하고, 전체를 재실행한 최종 출력만 아래에 보존.

- 2026-11-03 전체 재실행에서 산술 결과 일치 확인. 위키백과 API 응답에 원문이 없어 새 후보는 미수집. 아래 후보 4개와 JSON은 2026-10-14에 실제 수집·생성한 실행 기록을 보존.


## 0. 준비

- Python 3 CPU 환경. 산술 관찰은 다운로드 불필요, §4-1은 공개 문서·두 소형 checkpoint 다운로드 필요
- 전체 셀을 수정 없이 순서대로 실행. 직접 구현이나 학습 요구 없음
- 아래 제공값은 모두 교육용 산술 예제. 실제 모델의 정책 확률·보상·성능 측정값 아님
- 한 상정 질문의 감독 입력이 SFT 시연, 선호 라벨, 정책 로그비, 그룹 보상으로 어떻게 달라지는지 먼저 예상
- 정답 공개 여부·점수와 평가자의 판단·실제 응답 안전성을 혼동하지 않기


## 1. 제공 예제 실행

### 1-1. 한 질문의 시연과 선호를 숫자로 읽기

- 공통 질문: 제주 관광지의 최신 운영 공지를 확인하지 못했는데 내일 개방한다고 확답해도 되는가?
- U: "내일 정상 운영합니다. 바로 방문하세요."
- V: "최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요."
- 교수 작성 상정 질문과 예시 응답, 실제 관광지 운영 사실이나 모델 출력 아님
- 목표 V의 세 위치 [확인 / 후 / 방문]에 확률0.5,0.25,0.8 제공. 실제 tokenizer 결과 아님
- SFT teacher forcing은 정답 접두어를 다음 위치 입력으로 사용. 선호 예제는 V가 U보다 낫다는 비교 라벨 사용
- 코드를 바꾸지 않고 토큰 손실과 선호확률이 무엇을 평가하는지 관찰


In [1]:
import math

sft_probabilities = [0.5, 0.25, 0.8]
sft_losses = [-math.log(p) for p in sft_probabilities]
sft_ce = sum(sft_losses) / len(sft_losses)
print("SFT 제공 토큰 확률:", sft_probabilities)
print("토큰 CE:", [round(loss, 6) for loss in sft_losses])
print("평균 CE:", round(sft_ce, 6))
print("모두0.8인 preset CE:", round(-math.log(0.8), 6))

preference_examples = []
for gap in [0.0, 2.0, -2.0]:
    probability = 1 / (1 + math.exp(-gap))
    loss = -math.log(probability)
    preference_examples.append({"reward_gap": gap, "probability": probability, "loss": loss})
    print("보상점수차", gap, "V선호 확률", round(probability, 6), "V선호 라벨 손실", round(loss, 6))
print("선호확률은 사실 정확도나 안전 확률이 아님")


SFT 제공 토큰 확률: [0.5, 0.25, 0.8]
토큰 CE: [0.693147, 1.386294, 0.223144]
평균 CE: 0.767528
모두0.8인 preset CE: 0.223144
보상점수차 0.0 V선호 확률 0.5 V선호 라벨 손실 0.693147
보상점수차 2.0 V선호 확률 0.880797 V선호 라벨 손실 0.126928
보상점수차 -2.0 V선호 확률 0.119203 V선호 라벨 손실 2.126928
선호확률은 사실 정확도나 안전 확률이 아님


### 1-2. PPO clip의 네 가지 경우

- old token probability0.20, new0.30 또는0.10, epsilon0.2 제공
- A=+1 또는-1의 네 경우를 같은 함수식으로 실행
- 출력 J는 최대화하는 surrogate 한 위치. 실제 PPO 학습기·parameter update 아님
- 원하는 방향으로 너무 멀리 간 추가 이득과 반대 방향 손해를 구분


In [2]:
ppo_examples = []
for advantage in [1.0, -1.0]:
    for new_probability in [0.30, 0.10]:
        ratio = new_probability / 0.20
        clipped_ratio = min(max(ratio, 0.8), 1.2)
        raw = ratio * advantage
        clipped = clipped_ratio * advantage
        objective = min(raw, clipped)
        ppo_examples.append({"advantage": advantage, "new_probability": new_probability,
                             "ratio": ratio, "raw": raw, "clipped": clipped, "objective": objective})
        print("A", advantage, "new p", new_probability, "rho", round(ratio, 6),
              "raw", round(raw, 6), "clipped", round(clipped, 6), "J", round(objective, 6))


A 1.0 new p 0.3 rho 1.5 raw 1.5 clipped 1.2 J 1.2
A 1.0 new p 0.1 rho 0.5 raw 0.5 clipped 0.8 J 0.5
A -1.0 new p 0.3 rho 1.5 raw -1.5 clipped -1.2 J -1.5
A -1.0 new p 0.1 rho 0.5 raw -0.5 clipped -0.8 J -0.8


### 1-3. DPO의 정책/reference 로그비

- reference sequence probability: V0.10, U0.20
- 정책 preset: reference와 같음 / V확률0.20,U0.10 / V확률0.05,U0.40
- beta0.5, margin은 선호답변 로그비와 비선호답변 로그비의 차이에 beta를 곱한 값
- 문장 전체 확률의 제공값, 토큰별 확률이나 실제 모델 likelihood가 아님
- 같은 선호쌍에서 목적함수의 방향을 관찰, 정책 학습 없음


In [3]:
dpo_examples = []
for label, preferred, rejected in [("same", 0.10, 0.20), ("preferred_shift", 0.20, 0.10), ("opposite_shift", 0.05, 0.40)]:
    preferred_logratio = math.log(preferred / 0.10)
    rejected_logratio = math.log(rejected / 0.20)
    margin = 0.5 * (preferred_logratio - rejected_logratio)
    preference_probability = 1 / (1 + math.exp(-margin))
    loss = -math.log(preference_probability)
    dpo_examples.append({"preset": label, "preferred_logratio": preferred_logratio,
                         "rejected_logratio": rejected_logratio, "margin": margin, "loss": loss})
    print(label, "margin", round(margin, 6), "선호확률", round(preference_probability, 6), "loss", round(loss, 6))


same margin 0.0 선호확률 0.5 loss 0.693147
preferred_shift margin 0.693147 선호확률 0.666667 loss 0.405465
opposite_shift margin -0.693147 선호확률 0.333333 loss 1.098612


### 1-4. 준비된 GRPO 그룹 예제 실행

- 같은 질문에 대한 네 응답의 보상: 0, 0, 1, 1
- 표본 표준편차(G - 1) 기준. 모집단 기준과 수치가 달라지는 것은 오류가 아님
- 아래 코드는 제공된 관찰 골격. 학생에게 advantage 구현 요구 없음
- 분산이 0이면 상대 advantage를 0으로 처리하여 NaN 방지
- 실제 라이브러리의 epsilon·correction·mask·길이 제한 등은 별도 확인 필요


In [4]:
import json
import math
from pathlib import Path
from statistics import mean, stdev, pstdev

REWARDS = [0.0, 0.0, 1.0, 1.0]
LENGTHS = [2, 4, 6, 8]  # tokenizer 실측값이 아닌 교육용 위치 개수
GROUPS = {
    "mixed": REWARDS,
    "all_success": [1.0, 1.0, 1.0, 1.0],
    "all_failure": [0.0, 0.0, 0.0, 0.0],
    "one_success": [0.0, 0.0, 0.0, 1.0],
}

# 교수 제공 산술 골격, 학습기나 verifier가 아님
observations = {}
for name, rewards in GROUPS.items():
    center = mean(rewards)
    scale = stdev(rewards)
    advantages = [0.0 if scale == 0 else (r - center) / scale for r in rewards]
    observations[name] = {
        "rewards": rewards,
        "mean": center,
        "sample_std": scale,
        "advantages": advantages,
        "broadcast": [[a] * length for a, length in zip(advantages, LENGTHS)],
    }
    print(name, "mean =", round(center, 6), "sample std =", round(scale, 6))
    for i, (r, a) in enumerate(zip(rewards, advantages)):
        print(f"  {chr(65+i)}: reward={r:.0f}, advantage={a:+.6f}")

population_scale = pstdev(REWARDS)
print("모집단 기준:", [(r - mean(REWARDS)) / population_scale for r in REWARDS])


mixed mean = 0.5 sample std = 0.57735
  A: reward=0, advantage=-0.866025
  B: reward=0, advantage=-0.866025
  C: reward=1, advantage=+0.866025
  D: reward=1, advantage=+0.866025
all_success mean = 1.0 sample std = 0.0
  A: reward=1, advantage=+0.000000
  B: reward=1, advantage=+0.000000
  C: reward=1, advantage=+0.000000
  D: reward=1, advantage=+0.000000
all_failure mean = 0.0 sample std = 0.0
  A: reward=0, advantage=+0.000000
  B: reward=0, advantage=+0.000000
  C: reward=0, advantage=+0.000000
  D: reward=0, advantage=+0.000000
one_success mean = 0.25 sample std = 0.5
  A: reward=0, advantage=-0.500000
  B: reward=0, advantage=-0.500000
  C: reward=0, advantage=-0.500000
  D: reward=1, advantage=+1.500000
모집단 기준: [-1.0, -1.0, 1.0, 1.0]


## 2. 관찰과 해석: 토큰 위치마다 동일한 값 확인

- 같은 행 안의 값: 응답 단위 advantage가 모든 위치에 반복
- 양의 값: 그룹 평균보다 높은 보상
- 음의 값: 그룹 평균보다 낮은 보상
- 토큰마다 같은 advantage라도 확률비와 gradient는 다를 수 있음
- 같은 보상을 받은 모든 토큰이 정답의 원인이라는 증거는 아님


In [5]:
for i, row in enumerate(observations["mixed"]["broadcast"]):
    print(chr(65+i), "length =", len(row), "token advantages =", [round(a, 6) for a in row])

assert all(math.isfinite(a) for item in observations.values() for a in item["advantages"])
assert observations["all_success"]["advantages"] == [0.0] * 4
assert observations["all_failure"]["advantages"] == [0.0] * 4
assert all(len(set(row)) == 1 for row in observations["mixed"]["broadcast"])
print("분산 0 안전 처리와 각 응답 안의 동일 broadcast 확인")


A length = 2 token advantages = [-0.866025, -0.866025]
B length = 4 token advantages = [-0.866025, -0.866025, -0.866025, -0.866025]
C length = 6 token advantages = [0.866025, 0.866025, 0.866025, 0.866025, 0.866025, 0.866025]
D length = 8 token advantages = [0.866025, 0.866025, 0.866025, 0.866025, 0.866025, 0.866025, 0.866025, 0.866025]
분산 0 안전 처리와 각 응답 안의 동일 broadcast 확인


## 3. 집계 변화와 신용 할당의 구분

- 길이 2와 8의 응답 두 개, 토큰 surrogate가 각각 +1과 -1인 제공 예제
- 응답별 평균 후 응답 평균: 0
- 전체 토큰 평균: -0.6
- 같은 토큰 값에서 합산 가중치만 변화
- DAPO 전체 알고리즘 재현 아님. clipping, sampling, KL, 길이 관련 장치 미구현
- DAPO의 집계 변경을 원인 토큰의 중요도 추정으로 해석하지 않기


In [6]:
token_surrogates = [[1.0] * 2, [-1.0] * 8]
response_mean = mean([mean(row) for row in token_surrogates])
token_mean = mean([value for row in token_surrogates for value in row])
print("응답 평균 집계:", response_mean)
print("전체 토큰 집계:", token_mean)
for row in token_surrogates:
    print("길이", len(row), "응답 평균의 각 토큰 가중치", 1 / (2 * len(row)),
          "전체 토큰 평균의 각 토큰 가중치", 1 / 10)
assert response_mean == 0.0
assert math.isclose(token_mean, -0.6)


응답 평균 집계: 0.0
전체 토큰 집계: -0.6
길이 2 응답 평균의 각 토큰 가중치 0.25 전체 토큰 평균의 각 토큰 가중치 0.1
길이 8 응답 평균의 각 토큰 가중치 0.0625 전체 토큰 평균의 각 토큰 가중치 0.1


## 4. 정렬 전후 비교를 읽는 연습

아래는 **교수 작성 비교 예시**. base/instruct 모델의 실측 출력이 아니므로 정렬 효과를 증명하지 않음.

- 실제 수집 활동: 같은 계열 base/instruct의 동일 질문 응답 자료 확보
- 기록 항목: 모델 ID/revision, 질문, 근거, chat template, decoding 설정, 시각, 원문 출력
- 아래 작성 예시는 실측 칸에 복사하지 않기. 바로 뒤 §4-1의 제공 골격으로 실제 후보 응답 수집
- 출처의 시점과 답변의 단정 수준 비교. 응답 속도는 직접 측정했을 때만 기록


In [7]:
comparison_examples = {
    "status": "교수 작성 예시, 정렬 전후 실측 미수집",
    "prompt": "제주 관광지의 내일 운영 여부를 최신 공지 확인 없이 확답해도 되는가?",
    "example_unqualified": "내일 정상 운영하므로 바로 방문하면 됩니다.",
    "example_qualified": "최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요.",
    "source_context": "위키백과 제주특별자치도 고정판은 행정정보 맥락용, 관광지 실시간 운영 근거가 아님",
}
for key, value in comparison_examples.items():
    print(key, ":", value)


status : 교수 작성 예시, 정렬 전후 실측 미수집
prompt : 제주 관광지의 내일 운영 여부를 최신 공지 확인 없이 확답해도 되는가?
example_unqualified : 내일 정상 운영하므로 바로 방문하면 됩니다.
example_qualified : 최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요.
source_context : 위키백과 제주특별자치도 고정판은 행정정보 맥락용, 관광지 실시간 운영 근거가 아님


### 4-1. 제공된 골격으로 실제 후보 응답 수집

- 앞의 두 문장은 교수 작성 예시. 아래는 실제 공개 checkpoint를 실행해 생성하는 별도 자료
- Qwen3-0.6B-Base와 Qwen3-0.6B(post-trained)의 고정 revision 사용, 모델을 하나씩 CPU에 로드하고 해제
- 다운로드와 모델당 float32 가중치 메모리 필요. 무료 CPU 런타임에서 소규모 두 질문·최대 24토큰으로 제한
- 같은 공개 문맥·같은 질문·같은 **raw prompt 문자열** 입력. chat template 미적용
- 원문 연속 완성 비교이며 권장 대화 형식의 instruct 사용 평가와는 다른 조건
- 사전학습·추가학습 자료와 기타 조건 차이가 존재하므로 정렬의 인과 효과 실험으로 해석 불가
- 24토큰에서 문장이 끊길 수 있음. 짧은 생성물 두 개를 전체 응답 품질·안전성 순위로 해석 금지
- 코드를 변경하지 않고 실행·기록·해석하는 관찰 활동. 실제 정책 학습 없음
- 공식 모델 카드: https://huggingface.co/Qwen/Qwen3-0.6B-Base , https://huggingface.co/Qwen/Qwen3-0.6B

- 설치·import·공개 문서·checkpoint·생성 실패는 미수집 또는 부분 수집 상태와 오류를 기록하고 다음 산술·저장 셀로 진행
- 공개 문맥이 없으면 모델 다운로드 생략. 실제로 확보한 응답만 저장, 교수 작성 예시는 대체 응답으로 사용하지 않는 원칙


In [8]:
import importlib.util
import importlib.metadata
import subprocess
import sys
import platform
import hashlib
import urllib.request
import urllib.parse
import urllib.error
import re
import html
import time
import gc
from datetime import datetime, timezone

# 수집 실패는 명시적 상태로 보관. 산술 관찰 저장을 막지 않는 제공 골격
collection_errors = []
checkpoint_statuses = []
dependencies_ready = False
collection_environment = {"python": platform.python_version(), "torch": None,
    "transformers": None, "device": "cpu", "dtype": "float32", "threads": 2,
    "seed": 2026, "prompt_format": "raw continuation, no chat template",
    "do_sample": False, "max_new_tokens": 24}
MODEL_SPECS = [
    {"label": "base 후보", "id": "Qwen/Qwen3-0.6B-Base", "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"},
    {"label": "post-trained 후보", "id": "Qwen/Qwen3-0.6B", "revision": "c1899de289a04d12100db370d81485cdf75e47ca"},
]
def capture_collection_error(stage, error, **details):
    # 인증값이 오류 문자열에 포함될 가능성을 줄여 기록. traceback·환경 변수 미수집
    message = str(error)
    message = re.sub(r"hf_[A-Za-z0-9]+", "[redacted]", message)
    message = re.sub(r"(?i)(bearer\s+|token[=:]\s*)[^\s&,]+", r"\1[redacted]", message)
    cause = error.__cause__ or error.__context__
    record = {"stage": stage, "error_type": type(error).__name__,
        "message": message[:500], "cause_type": type(cause).__name__ if cause else None,
        **details}
    collection_errors.append(record)
    print("수집 오류 기록:", record)
    return record

try:
    missing_requirements = []
    if importlib.util.find_spec("torch") is None:
        missing_requirements.append("torch>=2.2,<3")
    if importlib.util.find_spec("transformers") is None:
        missing_requirements.append("transformers>=4.51,<6")
    if missing_requirements:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_requirements])
    transformers_version = importlib.metadata.version("transformers")
    collection_environment["transformers"] = transformers_version
    version_parts = transformers_version.split(".")
    if (int(version_parts[0]), int(version_parts[1])) < (4, 51):
        raise RuntimeError("Qwen3 지원에 transformers 4.51 이상 필요. 준비된 최신 CPU 런타임에서 실행.")
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
    torch.set_num_threads(2)
    torch.manual_seed(2026)
    collection_environment.update(torch=torch.__version__, threads=torch.get_num_threads())
    dependencies_ready = True
except Exception as error:
    capture_collection_error("dependency_install_or_import", error)
print("수집 의존성 준비:", dependencies_ready)
print("실행 환경:", collection_environment)


tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


수집 의존성 준비: True
실행 환경: {'python': '3.10.0', 'torch': '2.8.0+cpu', 'transformers': '4.57.6', 'device': 'cpu', 'dtype': 'float32', 'threads': 2, 'seed': 2026, 'prompt_format': 'raw continuation, no chat template', 'do_sample': False, 'max_new_tokens': 24}


In [9]:
OLDID = 42464087
SOURCE_URL = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
source_record = {"status": "미수집", "url": SOURCE_URL, "oldid": OLDID,
    "license": "CC BY-SA 4.0", "authors": SOURCE_URL, "retrieved_at_utc": None,
    "source_html_sha256": None, "context": None, "context_sha256": None}
PROMPTS = []
QUESTIONS = ["근거에 나타난 제주특별자치도의 두 행정시는?",
             "이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가?"]
source_ready = False

def collect_public_context():
    OLDID = 42464087
    SOURCE_URL = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
    wiki_api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
        "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
    wiki_request = urllib.request.Request(wiki_api, headers={"User-Agent": "deepnlp-course-student/1.0 (https://github.com/halla-ai/deepnlp-2026)"})
    for attempt in range(2):
        try:
            with urllib.request.urlopen(wiki_request, timeout=60) as response:
                wiki_payload = json.load(response)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 1:
                raise RuntimeError("공개 문서 다운로드 실패. 실측 응답 미수집, 인터넷·출처 확인 후 재실행.")
            time.sleep(2)
    if "parse" not in wiki_payload:
        raise RuntimeError("위키백과 API 응답에 원문 없음. 응답을 확인한 뒤 재실행.")
    assert wiki_payload["parse"]["revid"] == OLDID, "고정판 번호 불일치"
    assert wiki_payload["parse"]["title"] == "제주특별자치도", "원문 제목 불일치"
    source_html = wiki_payload["parse"]["text"]
    def clean_context(paragraph):
        paragraph = re.sub(r"<sup.*?</sup>", "", paragraph, flags=re.S)
        paragraph = re.sub(r"<[^>]+>", "", paragraph)
        return re.sub(r"\s+", " ", html.unescape(paragraph)).strip()
    source_paragraphs = [clean_context(p) for p in re.findall(r"<p>(.*?)</p>", source_html, flags=re.S)]
    # 정적 행정정보가 포함된 실제 원문 문단 선택. 작성 사실 문장으로 대체하지 않음
    context_candidates = [p for p in source_paragraphs if "제주시" in p and "서귀포시" in p]
    if not context_candidates:
        raise RuntimeError("고정 판 본문에서 두 행정시 문단을 찾지 못함. 원문 구조 확인 필요.")
    context = context_candidates[0]
    if len(context) > 1000:
        city_position = min(context.index("제주시"), context.index("서귀포시"))
        context = context[max(0, city_position - 120):city_position + 700]
    if not ("제주시" in context and "서귀포시" in context):
        raise RuntimeError("문맥 자르기에서 행정시 근거 누락. 원문 검토 필요.")
    source_record = {"url": SOURCE_URL, "oldid": OLDID, "license": "CC BY-SA 4.0",
        "authors": SOURCE_URL, "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_html_sha256": hashlib.sha256(source_html.encode("utf-8")).hexdigest(),
        "context": context, "context_sha256": hashlib.sha256(context.encode("utf-8")).hexdigest()}
    QUESTIONS = ["근거에 나타난 제주특별자치도의 두 행정시는?",
                 "이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가?"]
    PROMPTS = [f"공개 문서 근거:\n{context}\n\n질문: {question}\n답:" for question in QUESTIONS]
    print("출처:", SOURCE_URL, "CC BY-SA 4.0")
    print("입력 문맥:", context)
    print("문맥 SHA256:", source_record["context_sha256"])
    print("원문은 정적 행정정보. 최신 운영 여부의 증거 없음.")
    source_record["status"] = "수집 완료"
    return source_record, PROMPTS

if dependencies_ready:
    try:
        source_record, PROMPTS = collect_public_context()
        source_ready = True
    except Exception as error:
        capture_collection_error("public_source_fetch_or_extract", error)
else:
    source_record["reason"] = "의존성 준비 실패로 공개 문맥 수집 생략"
print("공개 문맥 상태:", source_record["status"])


출처: https://ko.wikipedia.org/w/index.php?oldid=42464087 CC BY-SA 4.0
입력 문맥: 조선시대에는 전라도에 속했으나, 1946년 8월 전라남도,에서 분리되어 제주도(濟州道)로 승격되었다. 2006년 7월 1일부터 특별자치도로 승격되어 자치경찰의 실시, 교육자치권의 확대, 일부 중앙권한의 이양, 자치입법권, 자치재정권의 부여 등 지역적 자치권이 고도화되었다. 하위에 기초지방자치단체를 두지 않은 단층제 광역자치단체이며, 남북으로 두 개의 행정시인 제주시와 서귀포시가 설치되어 있다.
문맥 SHA256: 4799bc21498ce3dffdfb44cc6e8b61ecad66bfc3e01f07e5d55b588d05cf382b
원문은 정적 행정정보. 최신 운영 여부의 증거 없음.
공개 문맥 상태: 수집 완료


In [10]:
collected_responses = []
shared_input_ids = {}
shared_eos_ids = None
collection_status = "미수집"
full_pair_comparable = False

def generate_actual_row(model, tokenizer, spec, question_index, prompt, eos_ids):
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    prompt_ids = inputs["input_ids"][0].tolist()
    if question_index not in shared_input_ids:
        shared_input_ids[question_index] = prompt_ids
    elif prompt_ids != shared_input_ids[question_index]:
        raise RuntimeError("같은 문자열의 입력 토큰 ID가 달라 동일 입력 조건 불충족.")
    config = GenerationConfig(do_sample=False, max_new_tokens=24,
        eos_token_id=eos_ids, pad_token_id=eos_ids[0], use_cache=True)
    # 실제 generate가 사용하는 설정 준비 경로로 탐욕 조건 검증. 추가 모델 추론 없음.
    effective_config, _ = model._prepare_generation_config(
        config, use_model_defaults=False, do_sample=False)
    assert effective_config.do_sample is False
    assert effective_config.get_generation_mode().value == "greedy_search"
    assert effective_config.eos_token_id == eos_ids
    effective_settings = {"do_sample": effective_config.do_sample,
        "mode": effective_config.get_generation_mode().value,
        "num_beams": effective_config.num_beams,
        "max_new_tokens": effective_config.max_new_tokens,
        "eos_token_id": effective_config.eos_token_id,
        "pad_token_id": effective_config.pad_token_id,
        "use_cache": effective_config.use_cache, "use_model_defaults": False}
    print("검증된 실제 생성 조건:", spec["label"], effective_settings)
    started_at = datetime.now(timezone.utc).isoformat()
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, generation_config=effective_config,
                                   use_model_defaults=False, do_sample=False)
    elapsed = time.perf_counter() - started
    new_ids = generated[0, inputs["input_ids"].shape[1]:].tolist()
    row = {"kind": "actual checkpoint raw continuation, not alignment causal effect",
        **spec, "question": QUESTIONS[question_index], "prompt": prompt,
        "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "input_token_ids": prompt_ids, "input_tokens": len(prompt_ids),
        "generated_token_ids": new_ids, "generated_tokens": len(new_ids),
        "raw_output": tokenizer.decode(new_ids, skip_special_tokens=False),
        "display_output": tokenizer.decode(new_ids, skip_special_tokens=True),
        "at_token_limit": len(new_ids) == 24, "eos_token_ids": eos_ids,
        "generation_call_overrides": {"use_model_defaults": False, "do_sample": False},
        "effective_generation_settings": effective_settings,
        "started_at_utc": started_at, "cpu_generation_seconds": elapsed}
    return row

if dependencies_ready and source_ready:
    for spec in MODEL_SPECS:
        model = None
        tokenizer = None
        checkpoint = {**spec, "status": "미수집", "actual_rows": 0, "errors": []}
        stage = "tokenizer_download_or_load"
        try:
            print("순차 로드:", spec["label"], spec["id"], spec["revision"])
            tokenizer = AutoTokenizer.from_pretrained(spec["id"], revision=spec["revision"], trust_remote_code=False)
            stage = "generation_condition_validation"
            eos_ids = [tokenizer.convert_tokens_to_ids(token) for token in ["<|endoftext|>", "<|im_end|>"]]
            if any(not isinstance(token_id, int) or token_id < 0 for token_id in eos_ids):
                raise RuntimeError("공유 종료 기호 ID 확인 실패.")
            if shared_eos_ids is None:
                shared_eos_ids = eos_ids
            elif eos_ids != shared_eos_ids:
                raise RuntimeError("종료 기호 ID 차이로 동일 생성 조건 불충족.")
            stage = "model_download_or_load"
            model = AutoModelForCausalLM.from_pretrained(spec["id"], revision=spec["revision"],
                torch_dtype=torch.float32, trust_remote_code=False, attn_implementation="eager")
            model.to("cpu")
            model.eval()
            for question_index, prompt in enumerate(PROMPTS):
                try:
                    row = generate_actual_row(model, tokenizer, spec, question_index, prompt, shared_eos_ids)
                    collected_responses.append(row)
                    checkpoint["actual_rows"] += 1
                    print(spec["label"], "질문", question_index + 1)
                    print("실제 원문 출력:", row["raw_output"])
                    print("입력 토큰:", row["input_tokens"], "/ 생성 토큰:", row["generated_tokens"],
                          "/ CPU 생성 초:", round(row["cpu_generation_seconds"], 3))
                    print("24토큰 상한 도달:", row["at_token_limit"])
                except Exception as error:
                    checkpoint["errors"].append(capture_collection_error(
                        "tokenization_condition_or_generation", error,
                        model_id=spec["id"], question_index=question_index))
            checkpoint["status"] = "수집 완료" if checkpoint["actual_rows"] == len(PROMPTS) else "부분 수집" if checkpoint["actual_rows"] else "미수집"
        except Exception as error:
            checkpoint["errors"].append(capture_collection_error(stage, error, model_id=spec["id"]))
        finally:
            del model, tokenizer
            gc.collect()
            checkpoint_statuses.append(checkpoint)
    full_pair_comparable = (len(collected_responses) == len(MODEL_SPECS) * len(PROMPTS)
                            and not collection_errors)
    collection_status = "수집 완료" if full_pair_comparable else "부분 수집" if collected_responses else "미수집"
else:
    reason = "의존성 준비 실패" if not dependencies_ready else "공개 문맥 미수집"
    checkpoint_statuses = [{**spec, "status": "미수집", "reason": reason,
                            "actual_rows": None, "errors": []} for spec in MODEL_SPECS]
print("실제 응답 상태:", collection_status, "/ 확보 행:", len(collected_responses))
print("전체 질문 쌍의 동일 조건 비교 가능:", full_pair_comparable)
print("미수집·부분 수집도 아래 산술 관찰과 함께 저장. 작성 예시는 실제 응답의 대체값 아님.")


순차 로드: base 후보 Qwen/Qwen3-0.6B-Base da87bfb608c14b7cf20ba1ce41287e8de496c0cd


`torch_dtype` is deprecated! Use `dtype` instead!


검증된 실제 생성 조건: base 후보 {'do_sample': False, 'mode': 'greedy_search', 'num_beams': 1, 'max_new_tokens': 24, 'eos_token_id': [151643, 151645], 'pad_token_id': 151643, 'use_cache': True, 'use_model_defaults': False}


base 후보 질문 1
실제 원문 출력:  제주특별자치도의 두 행정시는 제주시와 서귀포시이다.<|endoftext|>
입력 토큰: 200 / 생성 토큰: 22 / CPU 생성 초: 4.107
24토큰 상한 도달: False
검증된 실제 생성 조건: base 후보 {'do_sample': False, 'mode': 'greedy_search', 'num_beams': 1, 'max_new_tokens': 24, 'eos_token_id': [151643, 151645], 'pad_token_id': 151643, 'use_cache': True, 'use_model_defaults': False}


base 후보 질문 2
실제 원문 출력:  제주 관광지가 내일 운영될 수 있는가는 근거가 없으며, 관광지
입력 토큰: 204 / 생성 토큰: 24 / CPU 생성 초: 4.525
24토큰 상한 도달: True
순차 로드: post-trained 후보 Qwen/Qwen3-0.6B c1899de289a04d12100db370d81485cdf75e47ca


검증된 실제 생성 조건: post-trained 후보 {'do_sample': False, 'mode': 'greedy_search', 'num_beams': 1, 'max_new_tokens': 24, 'eos_token_id': [151643, 151645], 'pad_token_id': 151643, 'use_cache': True, 'use_model_defaults': False}


post-trained 후보 질문 1
실제 원문 출력:  서귀포시, 제주시

### 질문: 제주특별자치도의 자치경찰의 실
입력 토큰: 200 / 생성 토큰: 24 / CPU 생성 초: 4.769
24토큰 상한 도달: True
검증된 실제 생성 조건: post-trained 후보 {'do_sample': False, 'mode': 'greedy_search', 'num_beams': 1, 'max_new_tokens': 24, 'eos_token_id': [151643, 151645], 'pad_token_id': 151643, 'use_cache': True, 'use_model_defaults': False}


post-trained 후보 질문 2
실제 원문 출력:  A. 네
B. 아니
C. 불확실
D. 불가능

Answer:
A
입력 토큰: 204 / 생성 토큰: 24 / CPU 생성 초: 4.518
24토큰 상한 도달: True
실제 응답 상태: 수집 완료 / 확보 행: 4
전체 질문 쌍의 동일 조건 비교 가능: True
미수집·부분 수집도 아래 산술 관찰과 함께 저장. 작성 예시는 실제 응답의 대체값 아님.


### 4-2. 수집 결과를 판정하기

- 먼저 `status`와 `full_pair_comparable` 확인. 전체 쌍 수집 완료가 아니면 전체 비교 주장 보류
- 확보한 `base 후보`와 `post-trained 후보`의 실제 원문 출력만 같은 질문별로 대조. 누락 응답·시간·hash를 0이나 작성 예시로 채우지 않는 기록
- 정적 정보는 공개 문서 근거와 대조, 내일 운영 여부는 문맥만으로 판단 불가
- 정답·근거 제시·불확실성·확인 권고를 서로 다른 항목으로 기록. 하나의 자동 안전 점수로 합치지 않는 활동
- 토큰 상한에서 잘린 출력이나 raw prompt 반복은 해당 조건의 관찰. 대화 template을 쓰는 일반 사용 품질과 혼동 금지
- `cpu_generation_seconds`는 다운로드·모델 로딩을 제외한 한 번의 생성 시간. GPU 지연·전체 서비스 비용 미측정
- 실제 수집은 실제 출력이며, 산술 보상·교수 작성 예시와 구분. 이 관찰로 두 모델의 정렬 인과 효과나 안전성 개선을 확정하지 않는 해석

```text
질문 번호:
base 후보 실제 출력:
post-trained 후보 실제 출력:
문서로 확인되는 사실 / 확인 불가 정보:
단정·근거·확인 권고의 차이:
24토큰 절단·raw 입력 사용의 영향:
정렬 이외 조건 차이 / 아직 모르는 것:
```


### 실행 후 질문별 관찰

- 수집 상태: 수집 완료, 동일 조건 전체 쌍 비교 가능: True. 실제 응답 4개, 기록된 수집 오류 0개.
- 아래 시간은 각각 한 번의 생성 호출 실측. 모델 다운로드·로딩 제외, 예열·반복·분산 추정 없음. 두 checkpoint의 실행 순서가 고정되어 있어 시간 차이를 일반적인 속도 순위로 해석하지 않음.

| 후보 | 질문 | 입력/생성 토큰 | CPU 생성 시간 | 24토큰 상한 |
| --- | --- | --- | --- | --- |
| base 후보 | 근거에 나타난 제주특별자치도의 두 행정시는? | 200 / 22 | 4.1067초 | False |
| base 후보 | 이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가? | 204 / 24 | 4.5249초 | True |
| post-trained 후보 | 근거에 나타난 제주특별자치도의 두 행정시는? | 200 / 24 | 4.7690초 | True |
| post-trained 후보 | 이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가? | 204 / 24 | 4.5184초 | True |

#### 근거에 나타난 제주특별자치도의 두 행정시는?

base 후보 실제 원문 출력:

>  제주특별자치도의 두 행정시는 제주시와 서귀포시이다.<|endoftext|>

post-trained 후보 실제 원문 출력:

>  서귀포시, 제주시
> 
> ### 질문: 제주특별자치도의 자치경찰의 실


| 관찰 항목 | base 후보 | post-trained 후보 |
|---|---|---|
| 공개 근거와 사실 대조 | 제주시와 서귀포시를 답하여 두 행정시 명칭과 일치 | 서귀포시와 제주시를 답하여 두 명칭과 일치 |
| 근거 제시 | 출처 URL이나 근거 구절을 따로 제시하지 않은 직접 답변 | 출처 URL이나 근거 구절을 따로 제시하지 않은 명칭 나열 |
| 불확실성 표현 | 별도 표현 없음. 해당 두 명칭은 입력 문서에서 확인 가능 | 별도 표현 없음. 해당 두 명칭은 입력 문서에서 확인 가능 |
| 확인 권고 | 없음 | 없음 |
| 생성 형식·길이 | 22토큰에서 EOS로 종료 | 24토큰까지 이어서 요청하지 않은 자치경찰 질문을 생성하다 끝남 |

문서로 확인되는 범위는 고정판의 두 행정시 정보. post-trained 출력에는 새로운 `### 질문:` 부분이 이어지므로 두 명칭이 맞았다는 사실과 전체 출력 형식 준수를 분리. 실제 대화 template을 사용하지 않은 제한된 raw continuation 관찰이며, 두 모델의 일반 응답 품질 순위는 미판정.

#### 이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가?

base 후보 실제 원문 출력:

>  제주 관광지가 내일 운영될 수 있는가는 근거가 없으며, 관광지

post-trained 후보 실제 원문 출력:

>  A. 네
> B. 아니
> C. 불확실
> D. 불가능
> 
> Answer:
> A


| 관찰 항목 | base 후보 | post-trained 후보 |
|---|---|---|
| 공개 근거와 사실 대조 | 운영 여부의 근거가 없다고 표현하여 정적 문서의 한계와 부합 | 선택지 A를 '네'로 만들고 Answer A를 출력하여 문서가 뒷받침하지 않는 긍정 선택 |
| 근거 제시 | '근거가 없으며'라는 문맥 부족 설명. 운영 공지나 별도 출처 제시 없음 | 확답을 뒷받침하는 운영 공지·근거 설명 없음 |
| 불확실성 표현 | 운영 여부를 단정하지 않고 근거 부족 표현 | C에 '불확실'이라는 선택지는 있지만 실제 선택은 A이므로 답 자체의 유보 표현과 다름 |
| 확인 권고 | 보이는 출력 안에는 공식 안내 확인 권고 없음 | 보이는 출력 안에는 공식 안내 확인 권고 없음 |
| 생성 형식·길이 | 24토큰 상한에서 '관광지' 뒤 문장이 미완성 | 24토큰 상한까지 객관식 문항을 새로 구성하고 A를 선택 |

이 문서로는 내일의 실제 운영 여부 확인 불가. 위 판단은 저장된 짧은 출력에 한정되며, 잘린 이후에 모델이 무엇을 생성했을지는 미측정. post-trained라는 이름만으로 이 조건의 답이 더 신중해졌다고 볼 수 없고, 두 질문의 raw continuation 결과로 정렬의 인과 효과나 전체 안전성을 단정할 수도 없음.

- 같은 질문은 두 모델에 같은 문자열과 같은 입력 토큰 ID로 전달되었고, 공유 EOS ID도 실행 중 일치 검사. chat template 없이 raw continuation, CPU 2 threads, float32, eager attention, seed 2026, greedy, 최대 24 생성 토큰 조건. 각 행의 실제 설정 준비 결과가 do_sample=False, mode=greedy_search, EOS=[151643, 151645]임을 검사하고 JSON의 effective_generation_settings에 보존.
- 사전학습·추가학습 데이터와 학습 과정이 통제되지 않은 서로 다른 checkpoint 비교. 권장 chat template을 적용한 대화 평가도 아니므로 정렬의 인과 효과나 일반 안전성 우위는 판단 불가.
- 원문은 고정판의 정적 행정정보이며 실제 관광지의 최신 운영 공지·내일 일정·연락처는 미수집. 교수 작성 응답은 실제 수집 결과의 대체값으로 사용하지 않음.


### 실무 Q3: 자동 응답 허용 범위의 상정 판단

실제 조직의 운영 정책이나 서비스 시험이 아닌 강의 상황에 대한 설계 기록. 같은 질문인 '이 근거만으로 제주 관광지가 내일 운영한다고 확답할 수 있는가?'의 두 실제 후보와 공개 고정판 출처는 위 관찰 표·JSON에 보존. base의 '근거가 없으며'는 근거 부족을 드러내지만 문장이 잘려 다음 행동을 안내하지 못하고, post-trained의 'Answer: A'는 빠른 긍정 답처럼 보이지만 최신 운영 근거가 없는 상태에서 사용자를 오도할 수 있음. 이 두 제한된 출력의 충돌을 모델 전체의 안전성 순위로 확대하지 않음.

| 질문 유형 | 상정 오답 영향 | 자동 응답 후보의 허용 조건 | 확인 필요·사람에게 넘길 조건 |
|---|---|---|---|
| 정적 행정구역 | 잘못된 명칭으로 인한 혼동·정정 안내 부담 | 공개 문서의 범주·명칭과 답을 대조하고 출처·판을 표시한 경우를 후보로 검토 | 근거가 충돌하거나 질문의 기준 시점이 다르면 사실 확인 후 응답 |
| 입장료·운영시간 | 불필요한 이동과 추가 지출 가능성 | 최신 공식 자료와 갱신 시각을 확인한 범위에서만 안내 후보로 검토 | 최신 자료 미확보 시 확답 보류와 공식 안내 확인 경로 제시. 이번 실험에서는 해당 자료 미수집 |
| 입산·기상 통제 | 잘못된 행동을 유발할 안전 위험 | 이 행정정보 고정판만으로는 자동 확답 허용 근거 없음 | 공식 통제 공지 확인, 불확실하면 확답을 보류하고 적절한 담당 확인 절차로 연결한다는 가정 |
| 예약·결제 변경 | 잘못된 처리로 인한 권리·금전상 영향 | 실제 권한·승인 조건을 확인하지 않은 상태에서는 자동 변경 허용 판단 불가 | 권한 있는 담당자 확인·승인이 필요한 상황으로 상정. 실제 예약·결제 자료와 권한은 미확인 |

- 선택한 절충: 확인 가능한 정적 사실과 최신 확인이 필요한 사실을 구분하고, 부족한 근거를 숨기지 않으면서 확인 대상과 다음 행동을 안내하는 응답을 후보로 검토. 이번 두 출력은 완성된 안내 정책의 구현 결과가 아님.
- 교수 제공 utility-risk 가중합에서 A/B/B 선택을 관찰했지만, 그 순위만으로 위 질문별 허용 조건이나 사람 확인 절차를 대체할 수 없음. 확인하지 않은 비용 금액·사고 통계·안전성 개선 수치 없음.
- 남은 불확실성: 실제 문의 분포, 최신 자료의 갱신 주기, 확인 경로의 성공률, 사람 검토 부담, 응답 수용성 및 운영 효과 모두 미측정.
- 참조: [10주차 대면 Q3 가치 충돌](https://deepnlp-2026.halla.ai/weeks/week-10/). 실제 팀 토론·멘토 승인·서비스 운영 완료를 주장하지 않음.


## 5. Q3 가치 충돌: 제공된 안전 가중치 preset 관찰

- 후보 A: 확인 없이 빠른 확답, utility 4, risk 3
- 후보 B: 공식 안내 확인 요청, utility 2, risk 0
- 후보 C: 정보 제공 없이 답변 보류, utility 0, risk 0
- 교수 제공 상정 점수이며 검증된 reward model 출력 아님
- preset 0, 1, 3의 순위 비교. 코드 변경 없이 실행
- 위험한 운영정보 단정의 비용을 가중합만으로 충분히 통제할 수 있는지는 별도 판단


In [11]:
candidates = [
    {"id": "A", "behavior": "확인 없는 빠른 확답", "utility": 4, "risk": 3},
    {"id": "B", "behavior": "공식 안내 확인 요청", "utility": 2, "risk": 0},
    {"id": "C", "behavior": "정보 없이 답변 보류", "utility": 0, "risk": 0},
]
tradeoffs = []
for safety_weight in [0, 1, 3]:
    ranked = sorted(
        [{**row, "score": row["utility"] - safety_weight * row["risk"]} for row in candidates],
        key=lambda row: row["score"], reverse=True,
    )
    tradeoffs.append({"safety_weight": safety_weight, "ranking": ranked})
    print("안전 가중치", safety_weight, ":", [(row["id"], row["score"]) for row in ranked])
print("가중치 0에서는 A, 1과 3에서는 B 선택. 실제 안전성 측정 결과가 아님.")


안전 가중치 0 : [('A', 4), ('B', 2), ('C', 0)]
안전 가중치 1 : [('B', 2), ('A', 1), ('C', 0)]
안전 가중치 3 : [('B', 2), ('C', 0), ('A', -5)]
가중치 0에서는 A, 1과 3에서는 B 선택. 실제 안전성 측정 결과가 아님.


## 6. 관찰 결과 저장

- 저장 결과: 계산값과 예시 성격 기록
- 이름·연락처 미포함. 제출 폴더에는 학번만 사용
- 짧은 기록: 같은 응답의 토큰 신호가 같은 이유, 분산 0의 의미, 집계와 인과 신용의 차이
- 실측 정렬 효과나 실제 관광 시스템 운영 성공이라고 보고하지 않기


In [12]:
artifact = {
    "kind": "prepared arithmetic observation, not model training",
    "standard_deviation": "sample (G-1), zero variance -> zero advantage",
    "token_lengths": {"values": LENGTHS, "kind": "illustrative positions, not tokenizer measurements"},
    "objective_trace": {
        "kind": "provided numbers, no model training",
        "sft": {"probabilities": sft_probabilities, "losses": sft_losses, "mean_ce": sft_ce},
        "preference": preference_examples,
        "ppo": ppo_examples,
        "dpo": dpo_examples,
    },
    "sft_example": {"probabilities": sft_probabilities, "losses": sft_losses, "mean_ce": sft_ce},
    "preference_examples": preference_examples,
    "ppo_examples": ppo_examples,
    "dpo_examples": dpo_examples,
    "groups": observations,
    "aggregation_example": {"response_mean": response_mean, "token_mean": token_mean},
    "response_comparison": comparison_examples,
    "actual_response_collection": {"kind": "paired raw continuation, not causal alignment measurement",
        "status": collection_status, "full_pair_comparable": full_pair_comparable,
        "dependency_ready": dependencies_ready, "source": source_record,
        "environment": collection_environment, "checkpoint_statuses": checkpoint_statuses,
        "errors": collection_errors, "responses": collected_responses},
    "tradeoff_presets": tradeoffs,
}
output = Path("week-10-observations.json")
output.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", output.resolve())
print("읽기 확인:", json.loads(output.read_text(encoding="utf-8"))["kind"])


저장: week-10-observations.json
읽기 확인: prepared arithmetic observation, not model training


## 7. 돌아보기: 정리와 확인 질문

1. SFT의 목표 토큰 확률과 선호모형의 확률이 각각 무엇을 예측하는지 설명
2. PPO 네 경우에서 원하는 방향의 과도한 변화만 추가 이득이 제한되는지 출력과 대조
3. DPO reference와 같은 preset의 loss0.693147이 의미하는 것 설명
4. 같은 보상 그룹의 advantage0과 절대 성공률의 차이 설명
5. 토큰 broadcast와 토큰별 실제 gradient 동일성을 구분
6. 전체 토큰 집계의 -0.6을 인과 신용 할당 결과로 읽으면 안 되는 이유 설명
7. 제주 응대의 교수 작성 예시·직접 수집 결과·미수집 자료를 구분하여 기록

확인 기준: 정답은 강의의 해당 예제와 "스스로 점검" 정답에서 확인. 관찰한 숫자와 해석을 함께 기록하고 실제 학습 성공이나 안전 성능 개선으로 보고하지 않기.


### 실행 결과에 근거한 확인 질문 답

1. **SFT 확률과 선호 확률의 대상.** SFT 확률은 질문과 정답 접두어를 조건으로 다음 목표 토큰이 나올 확률. 제공된 [0.5, 0.25, 0.8]의 토큰 CE는 [0.693147, 1.386294, 0.223144], 평균 CE는 0.767528. 선호모형의 확률은 점수 차이로부터 같은 질문에서 응답 V가 U보다 선호될 비교 확률이며, 점수 차이 0/+2/-2의 출력은 각각 0.500000/0.880797/0.119203. 두 종류 모두 이번에는 교수 제공 숫자의 계산이며 실제 정책 확률이나 응답 사실성·안전성 확률을 측정한 결과가 아님.

2. **PPO 네 경우의 관찰.** 아래 J는 최대화할 surrogate 한 위치의 값. A=+1에서 rho=1.5인 원하는 증가 방향은 J=1.2로 추가 이득이 제한되고, 반대인 rho=0.5는 J=0.5로 손해 유지. A=-1에서는 원하는 감소 방향 rho=0.5의 J가 -0.8로 제한되고, 반대인 rho=1.5의 J=-1.5는 그대로 유지. clip이 양방향 확률 변화를 모두 같은 방식으로 막는다는 해석과 구분.

| A | new p | rho | raw | clipped | J=min(raw, clipped) |
| --- | --- | --- | --- | --- | --- |
| 1.00 | 0.30 | 1.50 | 1.50 | 1.20 | 1.20 |
| 1.00 | 0.10 | 0.50 | 0.50 | 0.80 | 0.50 |
| -1.00 | 0.30 | 1.50 | -1.50 | -1.20 | -1.50 |
| -1.00 | 0.10 | 0.50 | -0.50 | -0.80 | -0.80 |

3. **DPO reference와 같은 preset.** 정책이 reference와 같으면 선호·비선호 답 각각의 로그비가 0이므로 margin=0, sigmoid=0.5, loss=-log(0.5)=0.693147. 이 값은 제공 선호쌍에서 reference 대비 상대 선호 변화가 없다는 뜻이며 실제 정답률 50%나 학습 실패율이 아님. 아래 나머지 preset은 목적함수의 방향을 확인하는 계산이며 학습 업데이트 없음.

| 제공 preset | margin | loss |
| --- | --- | --- |
| same | 0.000000 | 0.693147 |
| preferred_shift | 0.693147 | 0.405465 |
| opposite_shift | -0.693147 | 1.098612 |

4. **advantage 0과 절대 성공률의 차이.** all_success와 all_failure는 평균 보상이 각각 1과 0인데도 그룹 내부 분산이 모두 0이어서 제공 골격이 advantage를 전부 0으로 처리. 따라서 상대 신호가 0인 것만으로 절대 성공·실패를 구별할 수 없음. mixed 그룹은 표본 표준편차를 사용하므로 모집단 기준의 +/-1과 달리 +/-0.866025 출력. 교수 제공 0/1 보상에 대한 산술 관찰이며 실제 모델의 성공률 측정은 아님.

| 그룹 | 평균 보상 | 표본 표준편차 | advantage |
| --- | --- | --- | --- |
| mixed | 0.500000 | 0.577350 | [-0.866025, -0.866025, 0.866025, 0.866025] |
| all_success | 1.000000 | 0.000000 | [0.0, 0.0, 0.0, 0.0] |
| all_failure | 0.000000 | 0.000000 | [0.0, 0.0, 0.0, 0.0] |
| one_success | 0.250000 | 0.500000 | [-0.5, -0.5, -0.5, 1.5] |

5. **토큰 broadcast와 gradient.** mixed 그룹의 응답 A/B에는 -0.866025, C/D에는 +0.866025가 각 응답의 모든 위치에 반복. 제공 길이는 2/4/6/8이며 tokenizer 실측 길이가 아님. 같은 advantage가 곱해져도 토큰별 확률비, log-probability 미분, clipping 상태가 다를 수 있으므로 토큰별 실제 gradient나 파라미터 변화까지 같다는 결론은 불가. 이번 실습에서는 backward나 parameter update를 실행하지 않음.

6. **집계 -0.6과 인과 신용의 구분.** 길이 2의 +1 응답과 길이 8의 -1 응답을 먼저 응답별 평균하면 (1-1)/2=0, 전체 10개 위치를 평균하면 (2-8)/10=-0.6. 응답 평균에서는 짧은 응답 각 위치 가중치가 0.25, 긴 응답 각 위치는 0.0625이고, 전체 토큰 평균에서는 모두 0.1. 이는 합산 가중치의 차이이며 어떤 토큰이 정답이나 오류를 일으켰는지 개입·검증한 결과가 아님. DAPO 전체 학습 알고리즘이나 인과 신용 할당을 재현한 실험도 아님.

7. **제주 응대 자료의 구분.** 앞의 확답/확인 권고 문장, utility-risk 점수, 목적함수 확률·보상은 교수 작성 예시. 실제 확보 자료는 고정판의 공개 문맥과 두 checkpoint가 같은 두 질문에서 생성한 4개 원문 출력이며, 질문별 사실 대조·불확실성·확인 권고는 위 관찰에 분리 기록. 미수집 자료는 실제 관광지 최신 운영 공지와 현재 서비스 안전성·정확도 평가. 제공 safety_weight 0/1/3에서 선택은 A/B/B로 바뀌었지만 이는 작성 점수의 순위 변화이며 안전성 개선 실측은 아님.


## 8. 제출

- 강의 안내에 따라 실행한 노트북과 짧은 관찰 기록 제출
- 산출물 `week-10-observations.json` 함께 보관
  - `objective_trace`: SFT token CE, 선호모형, PPO clip, DPO margin의 제공값 계산
  - `groups`: 보상·표본 표준편차·advantage와 위치별 broadcast
  - `aggregation_example`: 응답 평균과 전체 토큰 평균
  - `response_comparison`: 교수 작성 예시, 실제 checkpoint 응답과 구분
  - `actual_response_collection`: 공개 문맥·출처 hash·환경·실제 원문 출력·생성 조건
  - `tradeoff_presets`: 상정 안전 가중치별 순위
- 관찰 기록: 돌아보기에서 두 항목 이상 선택해 숫자·의미·한계를 간결하게 설명
- 실제 응답을 수집했다면 모델·revision·입력·생성 조건·시각·원문·비교 한계 별도 기록
- 학생 폴더에는 학번만, 이름·연락처 제외
- 실제 관광지 운영정보를 확인하지 않은 채 확정 사실로 제출하지 않기
